# Resume the existing MSD MedSAM3 campaign

Run all cells in order on the same A100-SXM4-80GB type and software environment,
signed into the same Drive account. Keep the existing input files and
`results_matched200_noaug_108_v1` folder. No new ZIP or data upload is required:
this complete notebook includes the updated scheduler and uses the already
uploaded `medsam3_msd_108_fast_eval_code.zip`, `medsam3_msd_data.zip` and `sam3.pt`.

The previous session stopped on a temporary Drive API rate limit. Quota checks now
retry with capped exponential backoff (up to 10 retries) while existing workers
continue. New workers still require a successful fresh storage check. Persistent
errors trigger the original clean checkpoint shutdown. Manual STOP and the
22-hour session stop are still respected during retries.

Legacy recovery archives omitted the training-history JSON sidecar. This version
restores a missing sidecar from the exact history already saved inside the
checkpoint, after checking its protocol, update ledger and completion hash.
It does not replay updates or redo completed evaluation. Future clean-stop
supplements also include this sidecar.

Completed experiments are verified and skipped. Interrupted evaluation resumes
from its saved checkpoint and completed volumes. Matching calibration and
preflight are reused. The original 108 task identities, 200-update recipe,
no augmentation, medical adapter and LoRA weights remain unchanged. Bands and
bands+edge are not repeated; the 27 historical results remain reference only.
Use only one runtime writing this results folder. This is a continuation, not
another set of 108 runs.


## Evaluation acceleration and verification

The original evaluator encodes each 1008×1008 slice twice, once per class prompt.
This version can encode it once, reuse only its prompt-independent image features,
and reuse the two text encodings within this trained checkpoint. Both prompt-conditioned
decoders, all slices, both classes, image resolution, BF16 precision and volume metrics
remain the same. No augmentation, batching change, extra experiments or optimizer updates.

When a run reaches evaluation, the **first unfinished volume** is compared slice by slice
against the original path. The original predictions produce that volume's metrics.
The faster path is enabled only if all predicted labels are identical, log probabilities
pass the existing strict tolerances (rtol 1e-5, atol 1e-7), RNG remains unchanged, and
measured slice time improves by at least 2%. The first slice of each subsequent volume
is checked again. A failed comparison or absent speed gain automatically keeps the
original evaluator for that checkpoint. No tolerance is relaxed to obtain a pass.

This adds one volume's candidate inference per checkpoint plus one checked slice per
remaining volume. There are no throwaway training runs or repeated calibration. GPU
speed is measured on your runtime; the package's local tests cannot establish A100 speed.
Look for `EVAL FAST` and the observed slice speedup in the worker's `train.log`; the
notebook shows `audit`, `fast`, or `original` and current slice progress every 30 seconds.

Completed archives and already evaluated volumes are skipped. A clean stop preserves
partial evaluation. The audit is reused only for the same checkpoint, code and environment.
Receipts and extension source are saved under `RESULTS/evaluation_extension/`; the frozen
training recipe and archive layout stay unchanged. The feature caches are scoped to one
checkpoint and are never used during training or calibration.



## 1. Connect to the same Drive results


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time
NOTEBOOK_STARTED = globals().get('NOTEBOOK_STARTED', time.monotonic())
HOME_DIR = Path('/content/drive/MyDrive/hippo_medsam3_msd_extension')
INPUT = HOME_DIR/'input'
RESULTS = HOME_DIR/'results_matched200_noaug_108_v1'
BUNDLE = Path('/content/hippo_medsam3_msd_108_bundle')
CONTROLLER = Path('/content/hippo_medsam3_parallel_fast_eval_v1')
RESULTS.mkdir(parents=True, exist_ok=True)
print('Preserving and resuming:', RESULTS)


## 2. Verify and stage without replacing work
The existing scheduler lock must be free. Unchanged files are reused; differing files cause
a stop instead of replacement. Existing `work/`, recovery, archives and source bindings stay
intact. On a new VM only, this restores code and data from the existing uploads.


In [ ]:
def stage_parallel(package, data_zip, base_file, bundle, controller, results,
                   package_sha, manifest_sha, data_sha, base_sha):
    import fcntl
    import hashlib
    import json
    from pathlib import Path
    import shutil
    import zipfile

    package, data_zip, base_file, bundle, controller, results = map(
        Path, (package, data_zip, base_file, bundle, controller, results))

    def sha(path):
        h = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(8*1024**2), b''):
                h.update(block)
        return h.hexdigest()

    def safe(name):
        p = Path(name)
        if p.is_absolute() or '..' in p.parts:
            raise ValueError('Unsafe package path')
        return p

    def install_bytes(target, value, expected):
        if hashlib.sha256(value).hexdigest() != expected:
            raise ValueError('Package member hash mismatch: '+str(target))
        if target.is_symlink():
            raise ValueError('Refusing to overwrite symlink: '+str(target))
        if target.exists():
            if sha(target) != expected:
                raise ValueError('Existing file differs; preserved without replacement: '+str(target))
            return
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_name(target.name+'.staging')
        temp.write_bytes(value)
        temp.replace(target)

    if sha(package) != package_sha:
        raise ValueError('Upload hash mismatch: '+package.name)
    (bundle/'work').mkdir(parents=True, exist_ok=True)
    with (bundle/'work/sweep.lock').open('a') as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError('A sweep is active. Stop the old Run cell once and wait for its recovery upload before staging.') from None
        existing = bundle/'bundle_manifest.json'
        if existing.exists() and sha(existing) != manifest_sha:
            raise ValueError('Existing bundle identity differs. No work has been replaced.')
        with zipfile.ZipFile(package) as archive:
            names = archive.namelist()
            if len(names) != len(set(names)):
                raise ValueError('Duplicate package members')
            for info in archive.infolist():
                safe(info.filename)
                if (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('Symlink in package')
            package_manifest = json.loads(archive.read('package_manifest.json'))
            if set(names) != set(package_manifest)|{'package_manifest.json'}:
                raise ValueError('Package file list differs')
            payload = archive.read('bundle/bundle_manifest.json')
            if hashlib.sha256(payload).hexdigest() != manifest_sha:
                raise ValueError('Scientific manifest differs')
            # Verify every existing scientific file before writing anything.
            for name, expected in package_manifest.items():
                part = safe(name)
                if part.parts[0] not in ('bundle', 'controller'):
                    raise ValueError('Unexpected package tree')
                target = (bundle if part.parts[0] == 'bundle' else controller)/Path(*part.parts[1:])
                if target.exists() and (target.is_symlink() or sha(target) != expected):
                    raise ValueError('Existing file differs; preserved: '+str(target))
            for name, expected in package_manifest.items():
                part = safe(name)
                target = (bundle if part.parts[0] == 'bundle' else controller)/Path(*part.parts[1:])
                install_bytes(target, archive.read(name), expected)
        data_manifest = json.loads((bundle/'data_manifest.json').read_text())
        missing = [name for name in data_manifest if not (bundle/'data'/safe(name)).exists()]
        for name, expected in data_manifest.items():
            path = bundle/'data'/safe(name)
            if path.exists() and sha(path) != expected:
                raise ValueError('Existing data changed: '+name)
        if missing:
            if sha(data_zip) != data_sha:
                raise ValueError('Data ZIP upload differs')
            with zipfile.ZipFile(data_zip) as archive:
                for name in missing:
                    install_bytes(bundle/'data'/safe(name), archive.read('data/'+name), data_manifest[name])
        base = bundle/'weights/sam3.pt'
        if base.exists():
            if sha(base) != base_sha:
                raise ValueError('Existing base checkpoint differs; preserved.')
        else:
            if shutil.disk_usage(bundle).free < 12*1024**3:
                raise RuntimeError('Need 12 GiB runtime disk space to stage the base checkpoint.')
            base.parent.mkdir(parents=True, exist_ok=True)
            temp = base.with_name('sam3.pt.staging')
            shutil.copyfile(base_file, temp)
            if sha(temp) != base_sha:
                raise ValueError('Base checkpoint copy differs')
            temp.replace(base)
        results.mkdir(parents=True, exist_ok=True)
        (results/'parallel_package.json').write_text(json.dumps(dict(
            package_sha256=package_sha, original_manifest_sha256=manifest_sha,
            science_source_changed=False, existing_results_preserved=True), indent=2))
    print('Original bundle and work preserved. Parallel supervisor staged separately.', flush=True)


stage_parallel(INPUT/'medsam3_msd_108_fast_eval_code.zip', INPUT/'medsam3_msd_data.zip', INPUT/'sam3.pt',
               BUNDLE, CONTROLLER, RESULTS, '4fef5c8191d394594f33c701bbf5e4c733a20cbfd5e8e2f2162a4efa42b3948b', '716f2e774fb9de3140baf1077eabff75fc0dc144f9a9e6752ec620dc5ddd6ac2',
               'ad869cf8c8862697b41bda740e91cae91bea6fcfdadca364b6f3ec5e6acf95e4', '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e')


# Resume-only scheduling fix; the frozen bundle and evaluation code stay identical.
resume_sources = {'parallel.py': '"""Adaptive process supervisor for the unchanged, frozen 108-run Colab bundle.\n\nScheduling, telemetry and supplementary recovery live here. An optional audited\nevaluation extension leaves training, task identities, optimizer and archives intact.\n"""\nimport argparse\nfrom collections import deque\nfrom dataclasses import dataclass, field\nimport fcntl\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport time\nimport uuid\n\nFROZEN_MANIFEST = \'716f2e774fb9de3140baf1077eabff75fc0dc144f9a9e6752ec620dc5ddd6ac2\'\nGIB = 1024**3\n\n\ndef transient_quota_error(error):\n    """Retry quota reads only for rate limits and transient HTTP server errors."""\n    status = getattr(getattr(error, \'resp\', None), \'status\', None)\n    if status in (429, 500, 502, 503, 504):\n        return True\n    if status != 403:\n        return False\n    try:\n        details = json.loads(error.content)[\'error\'][\'errors\']\n        return any(item.get(\'reason\') in (\'rateLimitExceeded\', \'userRateLimitExceeded\')\n                   for item in details)\n    except (AttributeError, ValueError, KeyError, TypeError):\n        return False\n\n\ndef load_runtime(root):\n    spec = importlib.util.spec_from_file_location(\'medsam3_frozen_runtime\', root/\'runtime.py\')\n    runtime = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(runtime)\n    if runtime.digest(root/\'bundle_manifest.json\') != FROZEN_MANIFEST:\n        raise ValueError(\'This supervisor requires the exact existing 108-run package.\')\n    runtime.activate(root)\n    return runtime\n\n\nclass SweepLock:\n    """Use the serial scheduler\'s lock too; never bypass an existing worker."""\n    def __init__(self, root):\n        self.path = Path(root)/\'work/sweep.lock\'\n        self.stream = None\n\n    def __enter__(self):\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.stream = self.path.open(\'a\')\n        try:\n            fcntl.flock(self.stream, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError:\n            self.stream.close()\n            raise RuntimeError(\'The previous sweep is still active. Stop its cell once and wait for recovery upload.\') from None\n        return self\n\n    def __exit__(self, *_):\n        self.stream.close()\n\n\ndef gpu_snapshot():\n    """Read resource telemetry without initializing a CUDA model in this process."""\n    args = [\'nvidia-smi\', \'--query-gpu=memory.total,memory.used,utilization.gpu\',\n            \'--format=csv,noheader,nounits\', \'--id=0\']\n    result = subprocess.run(args, capture_output=True, text=True, check=True, timeout=15)\n    total, used, utilization = map(float, result.stdout.strip().split(\',\'))\n    result = subprocess.run([\'nvidia-smi\', \'--query-compute-apps=pid,used_gpu_memory\',\n                             \'--format=csv,noheader,nounits\', \'--id=0\'],\n                            capture_output=True, text=True, check=True, timeout=15)\n    processes = {}\n    for line in result.stdout.splitlines():\n        fields = line.split(\',\')\n        if len(fields) == 2:\n            try:\n                processes[int(fields[0])] = float(fields[1]) / 1024\n            except ValueError:\n                pass\n    import psutil\n    try:\n        cpus = len(os.sched_getaffinity(0))\n    except AttributeError:\n        cpus = os.cpu_count() or 4\n    return dict(total_gib=total/1024, used_gib=used/1024, utilization=utilization,\n                processes=processes, host_free_gib=psutil.virtual_memory().available/GIB,\n                cpu_workers=max(1, cpus//4))\n\n\ndef progress(runtime, output):\n    ledger = output/\'checkpoint_ledger.json\'\n    step = runtime.read(ledger)[\'step\'] if ledger.exists() else 0\n    evaluation = output/\'evaluation.json\'\n    volumes = len(runtime.read(evaluation)[\'records\']) if evaluation.exists() else 0\n    phase = \'loading\' if not ledger.exists() else \'training\'\n    if step == 200:\n        phase = \'evaluating\' if (output/\'training_summary.json\').exists() else \'saving\'\n    if (output/\'completion.json\').exists():\n        phase = \'archiving\'\n    value = dict(step=step, volumes=volumes, phase=phase)\n    detail = output/\'evaluation_runtime.json\'\n    if phase == \'evaluating\' and detail.exists():\n        value[\'evaluation\'] = runtime.read(detail)\n    return value\n\n\n@dataclass\nclass Worker:\n    task: dict\n    process: object\n    log: object\n    started: float\n    initial: dict\n    peak_gib: float = 0.\n    state: dict = field(default_factory=dict)\n    stop_sent: bool = False\n    last_stop: float = -float(\'inf\')\n\n    def ready(self):\n        return (self.state.get(\'step\', 0) > self.initial[\'step\']\n                or self.state.get(\'volumes\', 0) > self.initial[\'volumes\'])\n\n\nclass AdaptiveLimit:\n    """Probe real pending work, comparing stable homogeneous throughput windows.\n\n    A mixed training/evaluation window cannot support a fair rate comparison.\n    In that case low utilization and progress can justify a bounded extra worker.\n    All observations and decisions are saved; no pilot run is thrown away.\n    """\n    def __init__(self, maximum=0, window=180):\n        self.maximum = maximum\n        self.window = window\n        self.target = 1\n        self.ceiling = None\n        self.memory_per_worker = 16.\n        self.samples = deque()\n        self.rates = {}\n        self.performance_limits = {}\n        self.last_change = -float(\'inf\')\n        self.last_signature = None\n\n    def resource_limit(self, gpu):\n        cap = min(gpu[\'cpu_workers\'], max(1, int((gpu[\'total_gib\']-8)/self.memory_per_worker)))\n        if self.maximum:\n            cap = min(cap, self.maximum)\n        if self.ceiling:\n            cap = min(cap, self.ceiling)\n        return max(1, cap)\n\n    def can_launch(self, gpu, workers):\n        # Reserve unseen peaks of current processes, not just their current VRAM.\n        reserve = sum(max(0., self.memory_per_worker-gpu[\'processes\'].get(w.process.pid, 0.)) for w in workers)\n        return (len(workers) < min(self.target, self.resource_limit(gpu))\n                and gpu[\'total_gib\']-gpu[\'used_gib\'] >= 8+self.memory_per_worker+reserve\n                and gpu[\'host_free_gib\'] >= 8+12*(1+sum(not w.ready() for w in workers))\n                and all(w.ready() and time.monotonic()-w.started >= 60 for w in workers))\n\n    def observe(self, now, gpu, workers):\n        for w in workers:\n            w.peak_gib = max(w.peak_gib, gpu[\'processes\'].get(w.process.pid, 0.))\n        # Never shrink the initial reservation until a worker has trained ten updates.\n        trained = [w.peak_gib for w in workers if w.state.get(\'step\', 0)-w.initial[\'step\'] >= 10 and w.peak_gib > 0]\n        if trained:\n            self.memory_per_worker = max(14., 1.25*max(trained), self.memory_per_worker if self.target > 1 else 0)\n        limit = self.resource_limit(gpu)\n        if self.target > limit:\n            self.target = limit\n            return \'Resource limit reduced; existing workers finish or checkpoint safely.\'\n        signature = tuple(sorted((w.task[\'id\'], w.state.get(\'phase\')) for w in workers))\n        if signature != self.last_signature:\n            self.samples.clear()\n            self.last_signature = signature\n        if not workers or len(workers) != self.target or any(not w.ready() for w in workers):\n            self.samples.clear()\n            return None\n        self.samples.append((now, sum(w.state[\'step\'] for w in workers),\n                             sum(w.state[\'volumes\'] for w in workers), gpu[\'utilization\']))\n        if now-self.samples[0][0] < self.window or now-self.last_change < self.window:\n            return None\n        first = self.samples[0]\n        dt = now-first[0]\n        phases = {w.state[\'phase\'] for w in workers}\n        arms = tuple(sorted({w.task[\'arm\'] for w in workers}))\n        average_gpu = sum(s[3] for s in self.samples)/len(self.samples)\n        key = None\n        if phases == {\'training\'} and self.samples[-1][1]-first[1] >= 10:\n            key, rate = (\'training\', arms), 60*(self.samples[-1][1]-first[1])/dt\n        elif phases == {\'evaluating\'} and self.samples[-1][2]-first[2] >= 3:\n            key, rate = (\'evaluation\', arms), 60*(self.samples[-1][2]-first[2])/dt\n        else:\n            rate = None\n        reason = None\n        if key is not None:\n            previous = self.rates.get(key)\n            measured_limit = self.performance_limits.get(key)\n            if measured_limit is not None and self.target >= measured_limit:\n                if self.target > measured_limit:\n                    self.target = measured_limit\n                    reason = f\'Reusing measured {key[0]} limit of {self.target} workers as active runs finish.\'\n            elif previous and previous[0] < self.target and rate < previous[1]*1.08:\n                self.target = previous[0]\n                self.performance_limits[key] = previous[0]\n                reason = f\'No clear throughput gain ({rate:.2f} vs {previous[1]:.2f}/min); returning to {self.target} workers as runs finish.\'\n            else:\n                self.rates[key] = (self.target, rate)\n                if self.target < limit:\n                    self.target += 1\n                    reason = f\'Measured {key[0]} throughput {rate:.2f}/min; trying {self.target} workers.\'\n        elif average_gpu < 80 and self.target < limit and (\n                self.samples[-1][1] > first[1] or self.samples[-1][2] > first[2]):\n            self.target += 1\n            reason = f\'Mixed phases, GPU average {average_gpu:.0f}%; trying {self.target} workers within resource reserves.\'\n        self.samples.clear()\n        if reason:\n            self.last_change = now\n        return reason\n\n\ndef verify_recovery(runtime, output, recovery):\n    """Confirm a clean stop uploaded the same ledger AND checkpoint as local SSD."""\n    runtime.check_ledger(output, recovery)\n    if not (output/\'checkpoint_latest.pt\').exists():\n        return\n    marker = runtime.read(recovery/\'latest.json\')\n    archive = recovery/runtime.relative(marker[\'file\'])\n    if runtime.digest(archive) != marker[\'sha256\']:\n        raise ValueError(\'Recovery archive hash mismatch after clean stop\')\n    with tarfile.open(archive) as tar:\n        ledger = json.load(tar.extractfile(\'checkpoint_ledger.json\'))\n        manifest = json.load(tar.extractfile(\'snapshot_manifest.json\'))\n    if (ledger != runtime.read(output/\'checkpoint_ledger.json\')\n            or manifest[\'checkpoint_latest.pt\'] != runtime.digest(output/\'checkpoint_latest.pt\')):\n        raise ValueError(\'Drive recovery is older than the local checkpoint; preserve this VM for recovery.\')\n\n\ndef save_supplement(runtime, root, durable, task):\n    """Retain completed evaluation volumes and logs when stopping between runs."""\n    output = root/\'work\'/task[\'id\']\n    if not (output/\'checkpoint_latest.pt\').exists():\n        return\n    target = durable/\'parallel_resume\'/task[\'id\']\n    files = {}\n    for name in (\'evaluation.json\', \'training_history.json\', \'training_summary.json\', \'slab_schedule.json\', \'runtime_recipe.json\', \'train.log\'):\n        if (output/name).exists():\n            runtime.copy_verified(output/name, target/name)\n            files[name] = runtime.digest(target/name)\n    runtime.write(target/\'manifest.json\', dict(bundle_sha256=FROZEN_MANIFEST, task=task,\n        checkpoint_sha256=runtime.digest(output/\'checkpoint_latest.pt\'),\n        ledger=runtime.read(output/\'checkpoint_ledger.json\'), files=files))\n\n\ndef restore_supplement(runtime, root, durable, task):\n    target = durable/\'parallel_resume\'/task[\'id\']\n    if not (target/\'manifest.json\').exists():\n        return\n    marker = runtime.read(target/\'manifest.json\')\n    output = root/\'work\'/task[\'id\']\n    if marker[\'task\'] != task or marker[\'bundle_sha256\'] != FROZEN_MANIFEST:\n        raise ValueError(\'Supplementary recovery identity differs\')\n    if marker[\'ledger\'] != runtime.read(output/\'checkpoint_ledger.json\'):\n        return  # A newer local checkpoint wins. Never roll it back.\n    if marker[\'checkpoint_sha256\'] != runtime.digest(output/\'checkpoint_latest.pt\'):\n        raise ValueError(\'Supplementary evaluation belongs to a different checkpoint\')\n    for name, sha in marker[\'files\'].items():\n        source = target/runtime.relative(name)\n        if runtime.digest(source) != sha:\n            raise ValueError(\'Supplementary recovery file changed\')\n        if not (output/name).exists():\n            runtime.copy_verified(source, output/name)\n\n\ndef restore_training_history(runtime, output, recovery):\n    """Recover a legacy omitted sidecar from the exact, ledger-bound checkpoint."""\n    path = output/\'training_history.json\'\n    checkpoint = output/\'checkpoint_latest.pt\'\n    if path.exists() or not checkpoint.exists():\n        return\n    runtime.check_ledger(output, recovery)\n    import torch\n    state = torch.load(checkpoint, map_location=\'cpu\', weights_only=False)\n    ledger = runtime.read(output/\'checkpoint_ledger.json\')\n    history = state[\'history\']\n    if (state[\'protocol_sha256\'] != runtime.digest(output/\'protocol.json\')\n            or state[\'protocol_sha256\'] != ledger[\'protocol_sha256\']\n            or state[\'step\'] != ledger[\'step\'] or len(history) != state[\'step\']\n            or [row[\'update\'] for row in history] != list(range(1, state[\'step\']+1))):\n        raise ValueError(\'Recovery checkpoint recipe, ledger and training history disagree\')\n    completion = output/\'completion.json\'\n    if completion.exists() and runtime.read(completion)[\'checkpoint_sha256\'] != runtime.digest(checkpoint):\n        raise ValueError(\'Completed evaluation belongs to a different checkpoint\')\n    runtime.write(path, history)\n\n\ndef prepare_task(runtime, root, durable, task):\n    output, recovery = root/\'work\'/task[\'id\'], durable/\'recovery\'/task[\'id\']\n    if not output.exists() or not any(output.iterdir()):\n        runtime.restore(output, recovery)\n    output.mkdir(exist_ok=True)\n    runtime.check_ledger(output, recovery)\n    if (output/\'checkpoint_latest.pt\').exists():\n        restore_supplement(runtime, root, durable, task)\n        restore_training_history(runtime, output, recovery)\n    return output, recovery\n\n\nclass Supervisor:\n    def __init__(self, runtime, root, durable, maximum=0, max_runs=108, hours=22, poll=5, telemetry=gpu_snapshot,\n                 fast_evaluation=False):\n        self.rt, self.root, self.durable = runtime, Path(root), Path(durable)\n        self.tuner, self.max_runs, self.poll, self.telemetry = AdaptiveLimit(maximum), max_runs, poll, telemetry\n        self.deadline = time.time()+hours*3600\n        self.active = {}\n        self.finished = set()\n        self.launched = 0\n        self.stopping = False\n        self.failure = None\n        self.previous_handlers = {}\n        self.session = time.strftime(\'%Y%m%d_%H%M%S\')+\'_\'+uuid.uuid4().hex[:8]\n        self.record = self.durable/\'parallel_sessions\'/self.session\n        self.last_display = -float(\'inf\')\n        self.fast_evaluation = fast_evaluation\n        self.quota_retry_at = 0.\n        self.quota_failures = 0\n\n    def event(self, message, **data):\n        print(message, flush=True)\n        self.record.mkdir(parents=True, exist_ok=True)\n        with (self.record/\'events.jsonl\').open(\'a\') as stream:\n            stream.write(json.dumps(dict(time=time.time(), message=message, **data))+\'\\n\')\n\n    def request_stop(self, *_):\n        self.stopping = True\n\n    def stop_workers(self):\n        for w in self.active.values():\n            ready = self.root/\'work\'/w.task[\'id\']/\'supervisor_ready.json\'\n            if (w.process.poll() is None and time.monotonic()-w.last_stop >= 5\n                    and ready.exists() and self.rt.read(ready)[\'pid\'] == w.process.pid):\n                w.process.send_signal(signal.SIGUSR1)\n                w.stop_sent = True\n                w.last_stop = time.monotonic()\n\n    def launch(self, task):\n        # Keep reaping/checkpointing active workers while a quota read backs off.\n        # Never use an old quota receipt to admit a new worker.\n        if self.stopping or time.monotonic() < self.quota_retry_at:\n            return\n        try:\n            self.rt.quota(self.durable)\n        except Exception as error:\n            if not transient_quota_error(error) or self.quota_failures >= 10:\n                raise\n            self.quota_failures += 1\n            delay = min(64., 2. ** self.quota_failures + uuid.uuid4().int % 1000 / 1000)\n            self.quota_retry_at = time.monotonic() + delay\n            self.event(f\'Drive quota check temporarily unavailable; retrying in {delay:.1f}s. Active workers continue.\',\n                       quota_retry=self.quota_failures, retry_seconds=delay)\n            return\n        if self.quota_failures:\n            self.event(\'Drive quota check recovered; continuing pending work.\')\n        self.quota_failures = 0\n        self.quota_retry_at = 0.\n        if self.stopping or (self.durable/\'STOP\').exists() or time.time() >= self.deadline-300:\n            self.stopping = True\n            return\n        quota = self.rt.read(self.durable/\'storage_quota.json\')\n        required = (8+2*(len(self.active)+1))*GIB\n        if \'limit\' in quota and int(quota[\'limit\'])-int(quota[\'usage\']) < required:\n            raise RuntimeError(\'Drive headroom insufficient for parallel recovery. Existing checkpoints are retained.\')\n        import shutil\n        if shutil.disk_usage(self.root).free < required:\n            raise RuntimeError(\'Runtime SSD headroom insufficient for parallel recovery.\')\n        output, _ = prepare_task(self.rt, self.root, self.durable, task)\n        args, env = self.rt.child(self.root, self.durable, \'train\', \'--task\', task[\'id\'], deadline=self.deadline-300)\n        # A tiny launcher ignores stop requests only during Python/module startup.\n        # The original engine then installs its own clean-stop handler unchanged.\n        # This prevents SIGUSR1 from killing a just-started process before import.\n        args = [args[0], str(Path(__file__).resolve()), \'_worker\', str(output/\'supervisor_ready.json\'), *args[1:]]\n        # Cache directories are non-scientific; keep font cache writes independent.\n        env[\'MPLCONFIGDIR\'] = str(output/\'matplotlib_cache\')\n        env[\'HIPPO_AUDITED_FAST_EVAL\'] = \'1\' if self.fast_evaluation else \'0\'\n        log = (output/\'train.log\').open(\'a\')\n        initial = progress(self.rt, output)\n        try:\n            process = subprocess.Popen(args, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n        except BaseException:\n            log.close()\n            raise\n        self.active[task[\'id\']] = Worker(task, process, log, time.monotonic(), initial, state=initial)\n        self.launched += 1\n        self.event(f"RUN {task[\'id\']} — resume update {initial[\'step\']}/200, evaluated {initial[\'volumes\']} volumes", pid=process.pid)\n\n    def reap(self):\n        for task_id, w in list(self.active.items()):\n            code = w.process.poll()\n            if code is None:\n                continue\n            w.log.close()\n            del self.active[task_id]\n            output, recovery = self.root/\'work\'/task_id, self.durable/\'recovery\'/task_id\n            if code == 0:\n                restore_training_history(self.rt, output, recovery)\n                self.rt.archive_run(self.root, output, self.durable, recovery, w.task)\n                self.finished.add(task_id)\n                self.event(\'ARCHIVED \'+task_id)\n            elif code == 75:\n                verify_recovery(self.rt, output, recovery)\n                save_supplement(self.rt, self.root, self.durable, w.task)\n                self.event(\'CHECKPOINT VERIFIED ON DRIVE \'+task_id, **progress(self.rt, output))\n                self.stopping = True\n            else:\n                self.rt.copy_verified(output/\'train.log\', self.durable/\'failures\'/task_id/\'train.log\')\n                self.failure = f\'{task_id} exited {code}. Recovery preserved. \'+(output/\'train.log\').read_text()[-4000:]\n                self.event(\'FAILED \'+task_id, returncode=code)\n                self.stopping = True\n\n    def inspect(self, gpu):\n        for task_id, w in self.active.items():\n            w.state = progress(self.rt, self.root/\'work\'/task_id)\n        now = time.monotonic()\n        reason = self.tuner.observe(now, gpu, list(self.active.values()))\n        if reason:\n            self.event(reason)\n        if now-self.last_display >= 30:\n            workers = [dict(id=w.task[\'id\'], pid=w.process.pid, peak_gib=round(w.peak_gib, 2), **w.state) for w in self.active.values()]\n            record = dict(time=time.time(), session=self.session, completed=len(self.finished), total=108,\n                target_workers=self.tuner.target, resource_limit=self.tuner.resource_limit(gpu),\n                gpu={k: v for k, v in gpu.items() if k != \'processes\'}, workers=workers, stopping=self.stopping)\n            self.rt.write(self.durable/\'parallel_status.json\', record)\n            with (self.record/\'telemetry.jsonl\').open(\'a\') as stream:\n                stream.write(json.dumps(record)+\'\\n\')\n            self.event(f"GPU {gpu[\'utilization\']:.0f}% | VRAM {gpu[\'used_gib\']:.1f}/{gpu[\'total_gib\']:.1f} GiB | workers {len(workers)}/{self.tuner.target} | archived {len(self.finished)}/108")\n            for w in workers:\n                detail = w.get(\'evaluation\')\n                suffix = (f" | {detail[\'backend\']}: {detail[\'case\']} slice {detail[\'slice\']}/{detail[\'slices\']}"\n                          if detail else \'\')\n                print(f"  {w[\'id\']}: {w[\'phase\']}, update {w[\'step\']}/200, evaluated {w[\'volumes\']} volumes{suffix}", flush=True)\n            self.last_display = now\n        if gpu[\'total_gib\']-gpu[\'used_gib\'] < 4 or gpu[\'host_free_gib\'] < 4:\n            self.event(\'Low memory headroom: saving all workers before continuing with a lower limit.\')\n            self.stopping = True\n            self.rt.write(self.durable/\'parallel_resource_limit.json\', dict(max_workers=max(1, len(self.active)-1), reason=\'Low memory headroom\'))\n\n    def run(self):\n        with SweepLock(self.root):\n            plan = self.rt.verify_bundle(self.root)\n            binding = self.rt.binding(self.root, self.durable)\n            if self.rt.read(self.durable/\'preflight.json\') != dict(status=\'passed\', **binding):\n                raise RuntimeError(\'The existing GPU preflight must be passed in this exact environment.\')\n            for fold in self.rt.FOLDS:\n                self.rt.fold_calibration(self.root, self.durable, fold)\n            for task in plan[\'tasks\']:\n                if self.rt.completed(self.root, self.durable, task):\n                    self.finished.add(task[\'id\'])\n                elif (self.root/\'work\'/task[\'id\']/\'completion.json\').exists():\n                    restore_training_history(self.rt, self.root/\'work\'/task[\'id\'], self.durable/\'recovery\'/task[\'id\'])\n                    self.rt.archive_run(self.root, self.root/\'work\'/task[\'id\'], self.durable, self.durable/\'recovery\'/task[\'id\'], task)\n                    self.finished.add(task[\'id\'])\n            # Interrupted local work takes priority, but all fresh baselines still finish first.\n            tasks = sorted(enumerate(plan[\'tasks\']), key=lambda it: (it[1][\'arm\'] != \'baseline\',\n                not (self.root/\'work\'/it[1][\'id\']/\'checkpoint_latest.pt\').exists()\n                    and not (self.durable/\'recovery\'/it[1][\'id\']/\'latest.json\').exists(), it[0]))\n            tasks = [t for _, t in tasks]\n            limit_file = self.durable/\'parallel_resource_limit.json\'\n            if limit_file.exists():\n                self.tuner.ceiling = self.rt.read(limit_file)[\'max_workers\']\n            self.event(f\'Preserved {len(self.finished)} verified completed runs. Same frozen engine, calibration and 108 task identities.\', bundle_sha256=FROZEN_MANIFEST)\n            if self.fast_evaluation:\n                self.event(\'Evaluation feature reuse is enabled only after a checkpoint-specific numerical and speed check. Training remains unchanged.\')\n            for sig in (signal.SIGINT, signal.SIGTERM, signal.SIGUSR1):\n                self.previous_handlers[sig] = signal.signal(sig, self.request_stop)\n            try:\n                while True:\n                    self.reap()\n                    if (self.durable/\'STOP\').exists() or time.time() >= self.deadline-300:\n                        self.stopping = True\n                    if self.stopping:\n                        self.stop_workers()\n                    if not self.active and (self.stopping or self.launched >= self.max_runs or len(self.finished) == len(tasks)):\n                        break\n                    gpu = self.telemetry()\n                    self.inspect(gpu)\n                    if self.stopping:\n                        self.stop_workers()\n                    elif self.launched < self.max_runs and self.tuner.can_launch(gpu, list(self.active.values())):\n                        pending = [t for t in tasks if t[\'id\'] not in self.finished and t[\'id\'] not in self.active]\n                        baselines = [t for t in pending if t[\'arm\'] == \'baseline\']\n                        if baselines:\n                            self.launch(baselines[0])\n                        elif not any(w.task[\'arm\'] == \'baseline\' for w in self.active.values()) and pending:\n                            self.launch(pending[0])\n                    elif not self.active and not self.stopping:\n                        raise RuntimeError(\'Not enough free GPU/host memory for the first worker plus reserves. Existing work is preserved.\')\n                    time.sleep(self.poll)\n            except BaseException as error:\n                self.failure = self.failure or f\'{type(error).__name__}: {error}\'\n                self.stopping = True\n            finally:\n                # Never release the sweep lock while an engine is still running.\n                if self.active:\n                    print(\'Saving every active worker to Drive. Keep this runtime connected until this cell returns.\', flush=True)\n                while self.active:\n                    self.stop_workers()\n                    try:\n                        self.reap()\n                    except Exception as error:\n                        self.failure = self.failure or f\'{type(error).__name__}: {error}\'\n                    if self.active:\n                        time.sleep(self.poll)\n                for sig, handler in self.previous_handlers.items():\n                    signal.signal(sig, handler)\n            result = self.rt.report(self.root, self.durable)\n            self.rt.write(self.record/\'result.json\', dict(status=\'failed\' if self.failure else \'complete\' if result[\'new_completed\'] == 108 else \'stopped\',\n                error=self.failure, new_completed=result[\'new_completed\'], launched=self.launched, bundle_sha256=FROZEN_MANIFEST))\n            if self.failure:\n                raise RuntimeError(self.failure)\n            self.event(\'All requested workers finished or stopped cleanly. Resume with the same Run cell and RESULTS folder.\')\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--bundle\', type=Path, required=True)\n    parser.add_argument(\'--durable\', type=Path, required=True)\n    parser.add_argument(\'--max-workers\', type=int, default=0, help=\'0: automatic, bounded by VRAM, CPU and measured throughput\')\n    parser.add_argument(\'--max-runs\', type=int, default=108)\n    parser.add_argument(\'--hours\', type=float, default=22)\n    parser.add_argument(\'--fast-evaluation\', action=\'store_true\', help=\'Audited evaluation-only image/text feature reuse\')\n    args = parser.parse_args()\n    if args.max_workers < 0 or not 1 <= args.max_runs <= 108 or args.hours <= 0:\n        parser.error(\'Invalid workers, runs or session duration\')\n    root, durable = args.bundle.resolve(), args.durable.resolve()\n    Supervisor(load_runtime(root), root, durable, args.max_workers, args.max_runs, args.hours,\n               fast_evaluation=args.fast_evaluation).run()\n\n\ndef launch_frozen_worker(marker, argv):\n    import runpy\n    signal.signal(signal.SIGUSR1, signal.SIG_IGN)\n    marker = Path(marker)\n    temporary = marker.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(dict(pid=os.getpid())))\n    temporary.replace(marker)\n    sys.argv = argv\n    sys.path.insert(0, str(Path(argv[0]).resolve().parent))\n    if os.environ.get(\'HIPPO_AUDITED_FAST_EVAL\') == \'1\':\n        spec = importlib.util.spec_from_file_location(\'medsam3_frozen_engine\', argv[0])\n        engine = importlib.util.module_from_spec(spec)\n        sys.modules[spec.name] = engine\n        spec.loader.exec_module(engine)\n        from fast_eval import install\n        durable = Path(argv[argv.index(\'--durable\')+1])\n        install(engine, durable)\n        engine.main()\n    else:\n        runpy.run_path(argv[0], run_name=\'__main__\')\n\n\nif __name__ == \'__main__\':\n    if len(sys.argv) > 1 and sys.argv[1] == \'_worker\':\n        launch_frozen_worker(sys.argv[2], sys.argv[3:])\n    else:\n        main()\n', 'fast_eval.py': '"""Checkpoint-local, audited feature reuse around the frozen evaluation loop.\n\nTraining, calibration, image sizes, per-prompt decoders and volume metrics are\nuntouched. A full-volume comparison gates reuse, followed by one checked slice\nper volume. Failed checks return to the original evaluator automatically.\n"""\nfrom contextlib import contextmanager, ExitStack\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\n\nRTOL, ATOL = 1e-5, 1e-7\n\n\ndef clone_tree(value, memo=None):\n    """Protect cached features from downstream mutation, retaining aliases."""\n    memo = {} if memo is None else memo\n    if id(value) in memo:\n        return memo[id(value)]\n    if isinstance(value, torch.Tensor):\n        result = value.clone()\n    elif isinstance(value, dict):\n        result = {k: clone_tree(v, memo) for k, v in value.items()}\n    elif isinstance(value, list):\n        result = [clone_tree(v, memo) for v in value]\n    elif isinstance(value, tuple):\n        result = tuple(clone_tree(v, memo) for v in value)\n    elif value is None or isinstance(value, (str, int, float, bool)):\n        return value\n    else:\n        raise TypeError(f\'Unsupported feature type: {type(value).__name__}\')\n    memo[id(value)] = result\n    return result\n\n\n@contextmanager\ndef replace_attribute(obj, name, value):\n    existed, previous = name in vars(obj), vars(obj).get(name)\n    setattr(obj, name, value)\n    try:\n        yield\n    finally:\n        if existed:\n            setattr(obj, name, previous)\n        else:\n            delattr(obj, name)\n\n\ndef equal_state(a, b):\n    if isinstance(a, torch.Tensor):\n        return isinstance(b, torch.Tensor) and torch.equal(a, b)\n    if isinstance(a, np.ndarray):\n        return isinstance(b, np.ndarray) and np.array_equal(a, b)\n    if isinstance(a, dict):\n        return isinstance(b, dict) and a.keys() == b.keys() and all(equal_state(a[k], b[k]) for k in a)\n    if isinstance(a, (list, tuple)):\n        return type(a) is type(b) and len(a) == len(b) and all(equal_state(x, y) for x, y in zip(a, b))\n    return a == b\n\n\nclass FeatureReuse:\n    """One image cache per slice; two text entries per fixed model instance."""\n    def __init__(self, model, processor, forward):\n        self.model, self.processor, self.forward = model, processor, forward\n        self.text = {}\n        self.counts = dict(image_encodes=0, image_hits=0, text_encodes=0, text_hits=0, transform_hits=0)\n\n    def clear(self):\n        self.text.clear()\n\n    def __call__(self, pixels):\n        if torch.is_grad_enabled() or any(m.training for m in self.model.modules()):\n            raise RuntimeError(\'Feature reuse is restricted to inference with model.eval().\')\n        if any(p.requires_grad for p in self.model.parameters()):\n            raise RuntimeError(\'Feature reuse requires frozen evaluation parameters.\')\n        backbone = self.model.backbone\n        image_fn, text_fn = backbone.forward_image, backbone.forward_text\n        transform = self.processor.transform\n        image_cache, transform_cache = [], []\n\n        def forward_image(samples):\n            if image_cache:\n                before, features = image_cache[0]\n                if samples.shape != before.shape or samples.dtype != before.dtype or not torch.equal(samples, before):\n                    raise ValueError(\'The two class prompts received different images.\')\n                self.counts[\'image_hits\'] += 1\n                return clone_tree(features)\n            result = image_fn(samples)\n            image_cache.append((samples.clone(), clone_tree(result)))\n            self.counts[\'image_encodes\'] += 1\n            return result\n\n        def forward_text(captions, input_boxes=None, additional_text=None, device=\'cuda\'):\n            # Unsupported prompt forms keep the original implementation.\n            if input_boxes is not None or additional_text is not None or not isinstance(captions, (list, tuple)):\n                return text_fn(captions, input_boxes=input_boxes, additional_text=additional_text, device=device)\n            key = (tuple(captions), str(device))\n            if key in self.text:\n                self.counts[\'text_hits\'] += 1\n                return clone_tree(self.text[key])\n            result = text_fn(captions, device=device)\n            if len(self.text) < 2:\n                self.text[key] = clone_tree(result)\n            self.counts[\'text_encodes\'] += 1\n            return result\n\n        def transform_once(value):\n            if transform_cache and torch.equal(value, transform_cache[0][0]):\n                self.counts[\'transform_hits\'] += 1\n                return clone_tree(transform_cache[0][1])\n            result = transform(value)\n            if not transform_cache:\n                transform_cache.append((value.clone(), clone_tree(result)))\n            return result\n\n        with ExitStack() as stack:\n            stack.enter_context(replace_attribute(backbone, \'forward_image\', forward_image))\n            stack.enter_context(replace_attribute(backbone, \'forward_text\', forward_text))\n            stack.enter_context(replace_attribute(self.processor, \'transform\', transform_once))\n            return self.forward(self.model, self.processor, pixels)\n\n\nclass EvaluationAudit:\n    def __init__(self, engine, original, model, processor, output, receipt, identity, minimum_gain=1.02):\n        self.engine, self.original, self.model, self.processor = engine, original, model, processor\n        self.output, self.receipt, self.identity = output, receipt, identity\n        self.forward = original.forward_plane\n        self.fast = FeatureReuse(model, processor, self.forward)\n        self.minimum_gain = minimum_gain\n        self.case, self.slice, self.depth = None, 0, 0\n        self.mode = \'audit\'\n        self.data = dict(identity=identity, status=\'audit\', full_volume=None, checked_slices=0,\n                         max_log_probability_difference=0., reference_seconds=0., candidate_seconds=0.,\n                         timed_slices=0, prediction_agreement=True, rng_agreement=True,\n                         sentinels=[], tolerance=dict(rtol=RTOL, atol=ATOL), fallback_reason=None)\n        if receipt.exists():\n            old = engine.read(receipt)\n            if old.get(\'identity\') == identity and old.get(\'status\') in (\'fast\', \'original\'):\n                self.data = old\n                self.mode = old[\'status\']\n                print(f\'EVAL: reusing {self.mode} decision for this exact checkpoint.\', flush=True)\n\n    def sync(self):\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n\n    def save(self):\n        self.data.update(status=self.mode, counts=self.fast.counts, updated_at=time.time())\n        self.engine.write(self.receipt, self.data)\n\n    def begin_volume(self, case, depth):\n        self.case, self.slice, self.depth = case, 0, depth\n        if self.mode == \'audit\':\n            self.data[\'full_volume\'] = dict(case=case, slices=depth, checked=0)\n            print(f\'EVAL AUDIT {case}: checking all {depth} slices; metrics use original predictions.\', flush=True)\n\n    def fallback(self, reason):\n        self.mode = \'original\'\n        self.data[\'fallback_reason\'] = reason\n        self.fast.clear()\n        self.save()\n        print(f\'EVAL: using original path. {reason}\', flush=True)\n\n    def checked(self, pixels):\n        before = self.engine.rng_state()\n        self.sync(); start = time.perf_counter()\n        reference = self.forward(self.model, self.processor, pixels)\n        self.sync(); reference_seconds = time.perf_counter()-start\n        after = self.engine.rng_state()\n        self.engine.set_rng(before)\n        try:\n            self.sync(); start = time.perf_counter()\n            candidate = self.fast(pixels)\n            self.sync(); candidate_seconds = time.perf_counter()-start\n            candidate_state = self.engine.rng_state()\n            if not equal_state(before, after) or not equal_state(after, candidate_state):\n                self.data[\'rng_agreement\'] = False\n                raise ValueError(\'Evaluation consumed RNG; feature reuse was disabled.\')\n            if reference[1] is not None or candidate[1] is not None:\n                raise ValueError(\'Evaluation unexpectedly returned a training loss.\')\n            a, b = reference[0], candidate[0]\n            if not torch.isfinite(a).all() or not torch.isfinite(b).all():\n                raise ValueError(\'Nonfinite audit output\')\n            difference = float((a-b).abs().max())\n            self.data[\'max_log_probability_difference\'] = max(self.data[\'max_log_probability_difference\'], difference)\n            # Compare the same exp/argmax used by the frozen volume evaluator.\n            if not torch.equal(a.exp().argmax(0), b.exp().argmax(0)):\n                self.data[\'prediction_agreement\'] = False\n                raise ValueError(\'Predicted labels changed.\')\n            torch.testing.assert_close(a, b, rtol=RTOL, atol=ATOL)\n            self.data[\'checked_slices\'] += 1\n            if self.mode == \'audit\':\n                self.data[\'full_volume\'][\'checked\'] += 1\n                # Exclude first two slices from timing to avoid cache/compile warm-up.\n                if self.slice >= 2:\n                    self.data[\'reference_seconds\'] += reference_seconds\n                    self.data[\'candidate_seconds\'] += candidate_seconds\n                    self.data[\'timed_slices\'] += 1\n            else:\n                self.data[\'sentinels\'].append(dict(case=self.case, slice=self.slice, max_difference=difference))\n        except Exception as error:\n            self.fallback(f\'{type(error).__name__}: {str(error)[:1200]}\')\n        finally:\n            self.engine.set_rng(after)\n        # All pilot and sentinel predictions remain the original predictions.\n        return reference\n\n    def plane(self, model, processor, pixels, labels=None, criterion=None):\n        if model is not self.model or processor is not self.processor or labels is not None or criterion is not None:\n            raise ValueError(\'Evaluation wrapper called outside its fixed checkpoint/prompt scope.\')\n        if self.mode == \'audit\' or (self.mode == \'fast\' and self.slice == 0):\n            result = self.checked(pixels)\n        elif self.mode == \'fast\':\n            before = self.engine.rng_state()\n            try:\n                result = self.fast(pixels)\n            except Exception as error:\n                self.engine.set_rng(before)\n                self.fallback(f\'{type(error).__name__}: {str(error)[:1200]}\')\n                result = self.forward(model, processor, pixels)\n        else:\n            result = self.forward(model, processor, pixels)\n        self.slice += 1\n        if self.mode == \'audit\' and self.slice == self.depth:\n            ratio = self.data[\'reference_seconds\']/max(self.data[\'candidate_seconds\'], 1e-12)\n            self.data[\'observed_slice_speedup\'] = ratio\n            if self.data[\'timed_slices\'] and ratio >= self.minimum_gain:\n                self.mode = \'fast\'\n                self.save()\n                print(f\'EVAL FAST: full-volume labels/log-probabilities passed; observed slice speedup {ratio:.2f}x.\', flush=True)\n            else:\n                self.fallback(f\'No measured slice-time improvement (observed {ratio:.2f}x).\')\n        # Local progress is cheap; durable receipts are updated once per volume.\n        self.engine.write(self.output/\'evaluation_runtime.json\', dict(case=self.case, slice=self.slice,\n            slices=self.depth, backend=self.mode, checkpoint_sha256=self.identity[\'checkpoint_sha256\']))\n        if self.slice == self.depth:\n            self.save()\n        return result\n\n\ndef install(engine, durable):\n    """Replace only the evaluate entry point; the frozen loop remains in use."""\n    evaluate = engine.evaluate\n    source = Path(__file__).resolve()\n    source_sha = engine.digest(source)\n\n    def audited_evaluate(original, model, processor, root, protocol, output):\n        destination = durable/\'evaluation_extension\'\n        engine.copy_verified(source, destination/\'source\'/f\'{source_sha}.py\')\n        identity = dict(extension_sha256=source_sha, checkpoint_sha256=engine.digest(output/\'checkpoint_latest.pt\'),\n            protocol_sha256=engine.digest(output/\'protocol.json\'), binding=engine.binding(root, durable),\n            task=output.name, policy=\'full-volume + per-volume first-slice check; original loop and metrics\')\n        key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()\n        receipt = destination/\'runs\'/output.name/f\'{key}.json\'\n        audit = EvaluationAudit(engine, original, model, processor, output, receipt, identity)\n        load_image = original.load_image\n\n        def tracked_load(*args, **kwargs):\n            value = load_image(*args, **kwargs)\n            case = args[1] if len(args) > 1 else kwargs[\'case\']\n            audit.begin_volume(case, value[1].shape[2])\n            return value\n\n        try:\n            with replace_attribute(original, \'load_image\', tracked_load), replace_attribute(original, \'forward_plane\', audit.plane):\n                return evaluate(original, model, processor, root, protocol, output)\n        finally:\n            audit.fast.clear()\n            if (output/\'evaluation.json\').exists():\n                audit.data[\'evaluation_sha256\'] = engine.digest(output/\'evaluation.json\')\n                audit.data[\'evaluated_volumes\'] = len(engine.read(output/\'evaluation.json\')[\'records\'])\n            audit.save()\n\n    engine.evaluate = audited_evaluate\n'}
resume_hashes = {'parallel.py': '3a316c84ae3bbcbae72ec4a6b24392d9dc620c72723cd2c40d42863a7c4dafdc', 'fast_eval.py': '7c0f55674e2381356dbaedfe77a224e81e0024d687cea781e9f988d2f638c339'}
resume_controller = Path('/content/hippo_medsam3_parallel_resume_v3')
import fcntl
with (BUNDLE/'work/sweep.lock').open('a') as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    for name, value in resume_sources.items():
        target = resume_controller/name
        expected = resume_hashes[name]
        if hashlib.sha256(value.encode()).hexdigest() != expected:
            raise RuntimeError('Resume source hash mismatch')
        if target.is_symlink() or (target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != expected):
            raise RuntimeError('Existing resume controller differs; preserving it: '+str(target))
    resume_controller.mkdir(parents=True, exist_ok=True)
    for name, value in resume_sources.items():
        target = resume_controller/name
        if not target.exists():
            temporary = target.with_suffix('.staging')
            temporary.write_text(value)
            temporary.replace(target)
    (RESULTS/'resume_controller.json').write_text(json.dumps(dict(files=resume_hashes,
        scientific_bundle_changed=False, augmentation='none', total_experiments=108), indent=2))
CONTROLLER = resume_controller
print('Drive quota retry controller ready. Completed work and scientific files preserved.')


## 3. Reuse the original environment and define notebook commands
An existing matching environment is reused without installation. A new VM installs the same
frozen requirements. Changing GPU/software after scientific work is blocked by the original
environment binding. Select the same A100 runtime type and High RAM.


In [ ]:
VENV = Path('/content/hippo_medsam3_venv')
PYTHON = VENV/'bin/python'
def setup(args):
    result = subprocess.run(list(map(str,args)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with (RESULTS/'parallel_setup.log').open('a') as log: log.write(result.stdout)
    print('\n'.join(result.stdout.splitlines()[-35:]))
    if result.returncode: raise RuntimeError('Environment setup failed; see parallel_setup.log')
if not PYTHON.exists():
    setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
probe = subprocess.run([str(PYTHON),'-m','pip','--version'], capture_output=True, text=True)
if probe.returncode:
    import urllib.request
    bootstrap = Path('/content/medsam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py', bootstrap)
    setup([PYTHON,bootstrap])
dependency_probe = """
import importlib.metadata as m, pathlib, sys
for line in pathlib.Path(sys.argv[1]).read_text().splitlines():
    if not line.strip(): continue
    parts = line.strip().split('==')
    try: version = m.version(parts[0])
    except m.PackageNotFoundError: sys.exit(1)
    if len(parts) == 2 and version != parts[1]: sys.exit(1)
"""
ready = subprocess.run([str(PYTHON),'-c',dependency_probe,str(BUNDLE/'requirements-colab.txt')]).returncode == 0
if not ready:
    setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
else:
    print('Reusing the existing dependency versions.')
def execute(args, log_name, parallel=False):
    process = subprocess.Popen(list(map(str,args)), start_new_session=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    interrupted = False
    with (RESULTS/log_name).open('a') as log:
        while True:
            try:
                for line in process.stdout:
                    print(line,end='',flush=True); log.write(line); log.flush()
                code = process.wait()
                break
            except KeyboardInterrupt:
                interrupted = True
                print('Clean stop requested. Keep this VM connected until every worker verifies its Drive recovery.')
                if process.poll() is None:
                    if parallel: process.send_signal(signal.SIGUSR1)
                    else: os.killpg(process.pid, signal.SIGINT)
    if code: raise RuntimeError(f'Command failed. See {RESULTS/log_name}. Existing work is retained.')
    return not interrupted
def campaign(command, *extra):
    return execute([PYTHON,BUNDLE/'runtime.py',command,'--bundle',BUNDLE,'--durable',RESULTS,*extra], command+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## 4. Reuse the passed preflight
The original command verifies the same bundle and environment. A matching passed preflight
returns immediately without repeating calibration or GPU diagnostics. An incomplete preflight
resumes only its missing work. Scientific runs cannot bypass this check.


In [ ]:
campaign('preflight')


## 5. Run or resume with automatic worker selection
`MAX_WORKERS = 0` means automatic; it does **not** mean two workers. The supervisor starts with
one pending experiment, measures progress, then tries additional workers within GPU, CPU, RAM
and storage reserves. On an 80 GB A100 it can consider more than two workers; the actual useful
number is measured on this runtime. No extra benchmark experiments are created.

It compares stable training/evaluation throughput windows and stops increasing when a measured
gain is too small. Mixed phases use GPU activity and headroom to decide whether to try another
worker. Startup is staggered; reduced limits drain running work instead of discarding updates.
Resource estimates are conservative, not a guarantee against OOM. Memory pressure stops workers
cleanly and records a lower ceiling for the next resume. A failed update is retained for recovery
audit instead of replayed silently.

Saved unfinished baselines resume first; all nine baselines complete before new constraint runs.
Every 30 seconds, output shows **GPU usage, worker count, update/200, evaluated volumes, current slice and evaluation backend**.
Full logs stay in `BUNDLE/work/<experiment>/train.log`. Status and tuning decisions are also
written to Drive (`parallel_status.json`, `parallel_sessions/`).

Interrupt this cell **once** to stop all workers. Wait until it returns: local checkpoints are
saved every update, periodic Drive recovery continues, and a clean stop verifies every final
recovery upload. Partly completed evaluation and logs are also saved on a clean stop. Completed
archives are verified and skipped; completed bulky recovery files are retired by the original code.
A manual `RESULTS/STOP` is respected and never removed automatically.

The 22-hour setting includes time spent in this notebook's setup cells. It does not know time
spent in a previous notebook: lower it accordingly if this VM is already old. Colab can stop
earlier; a hard VM loss with newer charged updates than the saved recovery requires an audit.
Leave this VM running until the cell has confirmed clean stops; do not start another notebook
or VM writing the same results concurrently. On resume, run this cell again on the same VM;
on a new VM, run all setup cells first.


In [ ]:
MAX_WORKERS = 0  # Automatic: resource and throughput limited, not fixed at two.
MAX_NEW_RUNS = 108  # Upper bound on starts; already completed experiments are skipped.
SESSION_HOURS = 22.0
remaining_hours = SESSION_HOURS - (time.monotonic()-NOTEBOOK_STARTED)/3600
if remaining_hours <= 0.1:
    raise RuntimeError('Notebook session budget reached. Resume in a fresh runtime after clean shutdown.')
execute([PYTHON, CONTROLLER/'parallel.py', '--bundle', BUNDLE, '--durable', RESULTS,
         '--max-workers', str(MAX_WORKERS), '--max-runs', str(MAX_NEW_RUNS),
         '--hours', str(remaining_hours), '--fast-evaluation'], 'parallel_run.log', parallel=True)


## 6. Results with fresh matched controls
Same original comparison: validation Dice and gains against the fresh baseline for each fold
and seed. Historical bands/bands+edge runs remain reference only. No conditions are added.


In [ ]:
campaign('report')
import pandas as pd
comparison = json.loads((RESULTS/'comparison.json').read_text())
display(pd.DataFrame(comparison['aggregate']))
display(pd.DataFrame(comparison['runs']))
